# 23 — Per-Category Feature-Shift Matrix (reviewer A: multi-class)

**Dijalankan di SageMaker.** Menjawab catatan reviewer JISA (A): tunjukkan
**matriks pergeseran fitur untuk 2–3 kategori serangan utama** (mis. DoS vs
Brute-force/Exploits). Untuk tiap kategori serangan, kita ukur jarak Wasserstein
$W_1$ ke-9 fitur SFM terhadap acuan **benign** (dalam ruang z-space gabungan),
lalu bandingkan antar-kategori. Ini membuktikan kuantitatif bahwa **kategori
serangan berbeda menggeser fitur berbeda** — dasar argumen kenapa kalibrasi
global tunggal tak cukup untuk multi-class (Limitations paper).

**Catatan kejujuran:** UNSW-NB15 punya kolom `attack_cat` (per-kategori mudah).
CIC `cleaned_100.pkl` labelnya sudah biner (kategori asli mungkin hilang saat
cleaning). Notebook mendeteksi ketersediaan kategori dan melapor apa adanya;
bila kategori CIC tak tersedia, analisis fokus pada UNSW (tetap menjawab reviewer:
shift antar-kategori dalam satu jaringan).

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('scipy','pandas','numpy','matplotlib','boto3') if u.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='catshift_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
# pemetaan kolom UNSW asli -> CANON (dur x1e6 utk us; sload/8; dload=dpkts/dur), sesuai nb21
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
print('=== SEL 1 (config) SELESAI ===')

## 2. Muat UNSW mentah dengan attack_cat + bangun 9 fitur

In [ ]:
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
UNS_CSV=first(['../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])
assert UNS_CSV, 'Butuh UNSW_NB15_training-set.csv (punya kolom attack_cat & label).'
u=pd.read_csv(UNS_CSV)
need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','label','attack_cat']
miss=[c for c in need if c not in u.columns]
assert not miss, f'kolom UNSW hilang: {miss}'
X=pd.DataFrame({'duration':pd.to_numeric(u['dur'],errors='coerce')*1e6,'fwd_pkts':u['spkts'],'bwd_pkts':u['dpkts'],
                'fwd_bytes':u['sbytes'],'bwd_bytes':u['dbytes'],'fwd_mean':u['smean'],'bwd_mean':u['dmean'],
                'src_load':pd.to_numeric(u['sload'],errors='coerce')/8.0,
                'dst_load':pd.to_numeric(u['dpkts'],errors='coerce')/pd.to_numeric(u['dur'],errors='coerce').replace(0,np.nan)})
X['label']=u['label'].astype(int)
X['cat']=u['attack_cat'].fillna('Normal').astype(str).str.strip()
X=X.replace([np.inf,-np.inf],np.nan).dropna()
print('UNSW rows:',len(X))
print('kategori:',X['cat'].value_counts().to_dict())
RESULTS['category_counts']={k:int(v) for k,v in X['cat'].value_counts().items()}
print('=== SEL 2 (muat UNSW + attack_cat) SELESAI ===')

## 3. Matriks $W_1$ per-kategori vs benign (z-space)
Untuk tiap kategori serangan dengan cukup sampel, hitung $W_1$ tiap fitur SFM
terhadap distribusi **benign** (Normal). Nilai besar = fitur itu bergeser jauh
dari benign untuk kategori tsb.

In [ ]:
MIN_N=300  # minimal sampel per kategori agar W1 stabil
benign=X[X['cat'].isin(['Normal'])][CANON]
if len(benign)==0: benign=X[X['label']==0][CANON]
# z-space fit pada seluruh data (agar fitur setara)
mu=X[CANON].mean(); sd=X[CANON].std().replace(0,1)
benz=(benign-mu)/sd
cats=[c for c,n in X['cat'].value_counts().items() if c not in ('Normal',) and n>=MIN_N]
print('kategori dianalisis (n>=%d):'%MIN_N, cats)
rows=[]
for c in cats:
    sub=((X[X['cat']==c][CANON]-mu)/sd)
    row={'category':c,'n':int(len(sub))}
    for f in CANON: row[f]=round(float(wasserstein_distance(sub[f].values, benz[f].values)),4)
    row['MEAN']=round(float(np.mean([row[f] for f in CANON])),4)
    rows.append(row)
mat=pd.DataFrame(rows).set_index('category')
import IPython.display as ipd; ipd.display(mat)
RESULTS['per_category_w1']=mat.reset_index().to_dict(orient='records')
print('=== SEL 3 (matriks W1 per-kategori) SELESAI ===')

## 4. Heatmap matriks kategori x fitur

In [ ]:
M=mat[CANON].values
fig,ax=plt.subplots(figsize=(9,0.6+0.5*len(mat)))
im=ax.imshow(M,aspect='auto',cmap='YlOrRd')
ax.set_xticks(range(len(CANON))); ax.set_xticklabels(CANON,rotation=45,ha='right')
ax.set_yticks(range(len(mat.index))); ax.set_yticklabels(mat.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]): ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=7)
fig.colorbar(im,ax=ax,label='$W_1$ vs benign')
ax.set_title('Per-category feature-shift matrix (UNSW-NB15, $W_1$ vs benign)')
plt.tight_layout(); savefig('category_shift_matrix.png')
print('=== SEL 4 (heatmap) SELESAI ===')

## 5. Simpan + UPLOAD S3

In [ ]:
jp=os.path.join(OUTDIR,'category_shift_results.json')
with open(jp,'w') as f: json.dump(RESULTS,f,indent=2)
mat.reset_index().to_csv(os.path.join(OUTDIR,'category_shift_matrix.csv'),index=False)
print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/catshift/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/catshift/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan + upload) SELESAI ===')
print('SEMUA SELESAI. Beri tahu asisten -> unduh s3://%s/%s/catshift/ untuk analisis.'%(S3_BUCKET,S3_PREFIX))